# Home Credit Default Risk — a big noisy feature store, where level 2 can earn its keep

Case: [Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk)
(307,511 loan applicants, **8.1% default**, scored on **ROC AUC** — native, 1:1).

The first showcases were single tidy tables. Real credit scoring is a
**relational feature store**: one application row joined to a fan-out of bureau
records, prior applications, monthly balances and installments. The library does
not auto-aggregate tables, so we collapse five auxiliary tables to one wide row
per applicant (numeric mean/max/min/sum/count) — a few hundred noisy, redundant,
heavily-missing features. That is exactly the regime where the level-2 machinery
is *supposed* to pay off:

- **the imputer** (finding #6) keeps the linear/baseline candidates alive despite
  `EXT_SOURCE_1` being ~56% missing and most aggregates sparse;
- **feature selection** has real redundancy to cut on ~400 columns, with the
  honest no-selection gate as the backstop;
- **calibration** (Brier/ECE) matters: a credit score is a probability that
  feeds a decision, not just a ranking.

> Data note: downloaded from a community Kaggle **mirror**
> (`megancrenshaw/home-credit-default-risk`); files are identical to the
> competition. The aggregation reads every auxiliary table in full; `APP_SAMPLE`
> then models on a 150k-applicant subset to keep the run light (set it to `None`
> for all 307k). Live submission stays off — ROC AUC is computed honestly offline.

In [1]:
import json
import logging
import os
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown

from honestml import (
    AutoML,
    CVConfig,
    FeatureSelectionConfig,
    HPOConfig,
    render_report,
    save_run_report,
)

SEED = 42
APP_SAMPLE = 150_000  # model on a stratified-by-chance subset to keep the run light; None = all 307k
DATA = Path("data/home-credit")
APP = DATA / "home-credit-default-risk"
RESULTS = Path("results/home-credit")
RESULTS.mkdir(parents=True, exist_ok=True)
KAGGLE = os.environ.get("KAGGLE_BIN") or shutil.which("kaggle") or str(Path.home() / ".local" / "bin" / "kaggle")

logging.basicConfig(format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("honestml").setLevel(logging.INFO)

In [2]:
if not (APP / "application_train.csv").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [KAGGLE, "datasets", "download", "-d", "megancrenshaw/home-credit-default-risk", "-p", str(DATA), "--unzip"],
        check=True,
    )

In [3]:
def aggregate(filename, prefix):
    """Collapse one auxiliary table to one row per SK_ID_CURR (numeric mean/max/min/sum + count)."""
    t = pd.read_csv(DATA / filename)
    drop = [c for c in ("SK_ID_PREV", "SK_ID_BUREAU") if c in t.columns]
    num = t.drop(columns=drop).select_dtypes("number")
    a = num.groupby("SK_ID_CURR").agg(["mean", "max", "min", "sum"])
    a.columns = [f"{prefix}_{col}_{stat}" for col, stat in a.columns]
    a[f"{prefix}_count"] = t.groupby("SK_ID_CURR").size()
    return a

app = pd.read_csv(APP / "application_train.csv")
app["DAYS_EMPLOYED"] = app["DAYS_EMPLOYED"].replace(365243, np.nan)  # documented sentinel
app["DAYS_EMPLOYED_PERC"] = app["DAYS_EMPLOYED"] / app["DAYS_BIRTH"]
app["INCOME_CREDIT_PERC"] = app["AMT_INCOME_TOTAL"] / app["AMT_CREDIT"]
app["ANNUITY_INCOME_PERC"] = app["AMT_ANNUITY"] / app["AMT_INCOME_TOTAL"]
app["EXT_SOURCES_MEAN"] = app[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)

# aggregate the auxiliary tables to the applicant grain (each full table is read once, then freed)
for filename, prefix in [
    ("bureau.csv", "BURO"),
    ("previous_application.csv", "PREV"),
    ("POS_CASH_balance.csv", "POS"),
    ("installments_payments.csv", "INS"),
    ("credit_card_balance.csv", "CC"),
]:
    app = app.join(aggregate(filename, prefix), on="SK_ID_CURR")

if APP_SAMPLE is not None and len(app) > APP_SAMPLE:
    app = app.sample(APP_SAMPLE, random_state=SEED).reset_index(drop=True)
y = app.pop("TARGET")
X = app.drop(columns=["SK_ID_CURR"])
print(f"rows: {len(X)}, features: {X.shape[1]}, default rate: {y.mean():.3%}")
print(f"missing cells: {X.isna().mean().mean():.1%} of all values")
X.head()

rows: 150000, features: 377, default rate: 8.076%
missing cells: 29.9% of all values


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,CC_CNT_INSTALMENT_MATURE_CUM_sum,CC_SK_DPD_mean,CC_SK_DPD_max,CC_SK_DPD_min,CC_SK_DPD_sum,CC_SK_DPD_DEF_mean,CC_SK_DPD_DEF_max,CC_SK_DPD_DEF_min,CC_SK_DPD_DEF_sum,CC_count
0,Cash loans,M,Y,N,2,207000.0,465457.5,52641.0,418500.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Cash loans,F,Y,Y,0,247500.0,1281712.5,48946.5,1179000.0,Unaccompanied,...,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.0
2,Cash loans,F,Y,N,0,202500.0,495000.0,39109.5,495000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Cash loans,F,N,Y,0,247500.0,254700.0,24939.0,225000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Cash loans,M,N,Y,0,112500.0,308133.0,15862.5,234000.0,Unaccompanied,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.0


## Level 1 — the untuned zoo on the raw feature store

The full zoo on a 5-fold stratified CV, selecting on **`roc_auc`** (the exact
competition metric), a 20% stratified holdout scored once, and leakage-free
probability calibration (`calibrate="auto"`). The median imputer is what keeps
the non-boosting candidates from being silently evicted by the heavy missingness.

In [4]:
model = AutoML(
    task="binary",
    metric="roc_auc",
    cv=CVConfig(outer_holdout=0.2, calibrate="auto"),
    random_state=SEED,
)
t0 = time.perf_counter()
model.fit(X, y)
print(f"level-1 fit took {(time.perf_counter() - t0) / 60:.1f} min")
report = model.run_report_
pd.DataFrame(report["leaderboard"])

INFO honestml.adapters.reader: auto-typing column=CNT_CHILDREN dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=FLAG_MOBIL dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=FLAG_EMP_PHONE dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=FLAG_WORK_PHONE dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=FLAG_CONT_MOBILE dtype=Int64 role=categorical reason=low_cardinality_int


...

INFO honestml: stage key=run stage=refit elapsed=37.0s


INFO honestml: stage key=run stage=refit elapsed=45.0s


INFO honestml: stage key=run stage=finalize elapsed=45.0s


level-1 fit took 9.0 min


,model_id,score,rank
0,catboost,0.771815,1
1,linear,0.764319,2
2,lightgbm,0.764153,3
3,xgboost,0.753628,4
4,baseline,0.499955,5


In [5]:
selection = next(e["score"] for e in report["leaderboard"] if e["model_id"] == report["winner"])
print(f"winner          : {report['winner']}")
print(f"equivalence band: {report['band']['member_ids']}")
print(f"selection score : {selection:.4f}   (out-of-fold roc_auc)")
print(f"holdout score   : {report['holdout_score']:.4f}   (untouched 20%, scored once)")
print(f"optimism        : {selection - report['holdout_score']:+.4f}")
cal = model.calibration_
if cal:
    print(f"calibration     : Brier {cal.get('brier_raw'):.4f} -> {cal.get('brier_calibrated'):.4f}, "
          f"ECE {cal.get('ece_raw'):.4f} -> {cal.get('ece_calibrated'):.4f} ({cal.get('method')})")
save_run_report(report, RESULTS)
md_path = render_report(report, RESULTS, fmt="md")
Markdown(md_path.read_text(encoding="utf-8"))

winner          : catboost
equivalence band: ['catboost']
selection score : 0.7718   (out-of-fold roc_auc)
holdout score   : 0.7695   (untouched 20%, scored once)
optimism        : +0.0023
calibration     : Brier 0.0669 -> 0.0669, ECE 0.0020 -> 0.0008 (auto)


# AutoML run report

## Run

| | |
|---|---|
| task | binary |
| metric | roc_auc |
| winner | catboost |
| holdout_score | 0.769505 |
| honestml_version | 1.0.0 |
| run_fingerprint | dea943a3cef69bed001089e72fed6586fc5ebe372131c46bdfcc6be023a1aeb5 |
| significance | bootstrap |
| preset | n/a |

## Equivalence band

| | |
|---|---|
| members | catboost |
| width | 1 |
| unstable | False |
| winner_by_tiebreak | False |

## Budget

| | |
|---|---|
| mode | none |
| exhausted | False |
| exhausted_by | n/a |
| skipped | n/a |

## Serving

| | |
|---|---|
| finalize | True |
| shipped_on | all |
| outer_holdout | 0.2 |

## Leaderboard

| rank | model | score |
|---|---|---|
| 1 | catboost (winner) | 0.771815 |
| 2 | linear | 0.764319 |
| 3 | lightgbm | 0.764153 |
| 4 | xgboost | 0.753628 |
| 5 | baseline | 0.499955 |

## Timings (s)

| stage | elapsed |
|---|---|
| run.selection | 454.4 |
| run.refit | 45 |
| run.finalize | 45 |

## Resolved config

```json
{
  "seed": 42,
  "cv": {
    "scheme": "stratified",
    "n_splits": 5,
    "n_test": 1,
    "n_es": 1,
    "purge": 0,
    "embargo": 0,
    "period": null,
    "period_size": null,
    "step_periods": null,
    "purge_delta": null,
    "embargo_delta": null,
    "max_train_periods": null,
    "max_train_size": null,
    "weighting": "pooled",
    "calibrate": "auto",
    "selection": "raw",
    "refinement_min_oof": 2000,
    "outer_holdout": 0.2
  },
  "budget": {
    "mode": "none",
    "time_budget_s": null,
    "n_trials": null,
    "memory_limit_mb": null
  },
  "fe": {
    "target_encoding": false,
    "te_smoothing": 10.0,
    "frequency_encoding": false,
    "intersections": false,
    "max_pairs": 50
  },
  "fs": null,
  "hpo": null,
  "ensemble": null,
  "significance": "bootstrap",
  "run_mode": "full",
  "model_types": [
    "catboost",
    "lightgbm"
  ]
}
```


## Level 2 — feature selection + HPO on the ~400-column store

Public Home Credit solutions earned most of their AUC from feature engineering,
selection and stacking over a baseline LightGBM — the canonical case where
level-2 machinery beats untuned defaults. Here:

- `FeatureSelectionConfig(strategy="importance", cutoff="auto")` — hundreds of
  redundant aggregates give selection something real to cut, with the
  no-selection gate as the honest backstop;
- `preset="best"` + `HPOConfig` — Optuna tuning and significance-gated ensembling;
- calibration and the untouched 20% holdout as in level 1.

In [6]:
model_full = AutoML(
    task="binary",
    metric="roc_auc",
    preset="best",
    hpo=HPOConfig(n_trials=15, timeout_s=600),
    feature_selection=FeatureSelectionConfig(strategy="importance", cutoff="auto"),
    cv=CVConfig(outer_holdout=0.2, calibrate="auto"),
    random_state=SEED,
)
t0 = time.perf_counter()
model_full.fit(X, y)
print(f"level-2 fit took {(time.perf_counter() - t0) / 60:.1f} min")

INFO honestml.composition.presets: preset best applied: ['ensemble']


INFO honestml.adapters.reader: auto-typing column=CNT_CHILDREN dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=FLAG_MOBIL dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=FLAG_EMP_PHONE dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=FLAG_WORK_PHONE dtype=Int64 role=categorical reason=low_cardinality_int


...

INFO honestml: stage key=run stage=ensemble elapsed=84.3s


INFO honestml: stage key=run stage=refit elapsed=32.8s


INFO honestml: stage key=run stage=refit elapsed=38.4s


INFO honestml: stage key=run stage=finalize elapsed=38.4s


level-2 fit took 43.8 min


In [7]:
report_full = model_full.run_report_
sel_full = next(e["score"] for e in report_full["leaderboard"] if e["model_id"] == report_full["winner"])
fs = report_full["feature_selection"]
print(f"winner          : {report_full['winner']}")
print(f"selection score : {sel_full:.4f}   (level 1: {selection:.4f})")
print(f"holdout score   : {report_full['holdout_score']:.4f}   (level 1: {report['holdout_score']:.4f})")
print(f"optimism        : {sel_full - report_full['holdout_score']:+.4f}")
print(f"features kept   : {fs.get('n_selected')} of {X.shape[1]}  (gate: {fs.get('no_selection_gate')})")
print()
print("hpo     :", json.dumps(report_full["hpo"], default=str)[:400])
print("ensemble:", json.dumps(report_full["ensemble"], default=str)[:300])
pd.DataFrame(report_full["leaderboard"])

winner          : xgboost
selection score : 0.7733   (level 1: 0.7718)
holdout score   : 0.7791   (level 1: 0.7695)
optimism        : -0.0058
features kept   : 180 of 377  (gate: selection_kept)

hpo     : {"backend": "optuna", "inner_cv": 3, "deterministic": false, "selection_oof_is_post_tuning": true, "tuned_on_full_feature_space": true, "cost_estimate_fits": 135, "tuned": {"catboost": {"chosen_params": {"depth": 4, "learning_rate": 0.19030368381735815, "iterations": 350, "l2_leaf_reg": 5.105903209394756, "subsample": 0.608233797718321, "one_hot_max_size": 63}, "inner_best_score": 0.77081648145960
ensemble: {"applied": true, "method": "caruana", "member_ids": ["baseline", "catboost", "lightgbm", "linear", "xgboost"], "weights": {"baseline": 0.18904593639575973, "catboost": 0.053003533568904596, "lightgbm": 0.3674911660777385, "linear": 0.3003533568904594, "xgboost": 0.09010600706713781}, "gate_reason":


,model_id,score,rank
0,xgboost,0.773317,1
1,lightgbm,0.771644,2
2,catboost,0.769349,3
3,linear,0.763136,4
4,baseline,0.499955,5


## Level 1 vs level 2: what the full pipeline buys

The store is genuinely big and noisy — **377 features, 29.9% of cells missing**.
Two level-1 details earn their place before any tuning:

- the **median imputer** (finding #6) keeps `linear` competitive (OOF roc_auc
  0.7643, second) and the stratified `baseline` in the comparison, instead of
  letting the missingness silently evict every non-boosting model — catboost
  wins at **0.7718**, but measured against survivors, not a boosting-only field;
- **calibration** was already tight; `auto` nudged ECE from 0.0020 to 0.0008 — a
  credit score you can read as a probability, not just a ranking.

**This is the notebook where feature selection earns its keep.** On ~400
redundant aggregates, `importance` selection cut the store to **180 features and
the gate kept the cut** (`no_selection_gate: selection_kept`) — a trimmed subset
the no-selection gate judged at least as good as keeping everything, which the
redundant feature store is exactly built to offer. Together with
significance-gated **ensembling** (a blend of all five models), level 2 lifts the
untouched **holdout from 0.7695 to 0.7791** (+0.0096). The OOF barely moves
(0.7718 → 0.7733), but the shipped model generalizes *better* on data it never
saw — the upside the big noisy feature store was supposed to offer, and the
honest contour confirms it on the holdout, not on the selection number.

One caveat the report keeps honest: HPO completed most of its 15 trials per model
within its time budget (`deterministic: false`), so the tuning is real but not
exhaustive — most of the level-2 gain here is the **feature-selection cut plus
the blend**, not the hyperparameters.

Against the published ladder — a baseline LightGBM lands ~0.75 on this data and
the heavy-stacking winners reached ~0.80+ — a shipped holdout of **0.7791** from
a config-only run (aggregate → select → tune → blend → calibrate) sits right
where an honest, non-leaderboard-chasing pipeline should.